In [ ]:
import sys, warnings, json
from pathlib import Path
import pandas as pd

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.data_loader import get_dataset
from src.features.graph_metrics import compute_graph_metrics
from src.evaluation.experiments import (
    run_node_classification_experiment,
    run_link_prediction_experiment,
)
from src.evaluation.community import run_louvain, evaluate_communities
from src.visualization.distributions import plot_degree_distribution

Path(PROJECT_ROOT / 'results').mkdir(exist_ok=True)

ds = get_dataset('cora', verbose=True)
G = ds['graph']
metrics = compute_graph_metrics(
    G, name='cora',
    metric_groups=['basic', 'degree', 'clustering', 'paths', 'community']
)

# Сохраняем метрики
pd.DataFrame([{k: v for k, v in metrics.items() if isinstance(v, (int, float, str))}]
             ).to_csv(PROJECT_ROOT / 'results' / 'demo_baseline_metrics.csv', index=False)

# Визуализация распределения степеней
plot_degree_distribution(G, name='Cora degree distribution', save=True, show=True)

# Node Classification
nc_res = run_node_classification_experiment(
    'cora', model='logistic', use_features=True, use_embeddings=False,
    dimensions=64, n_runs=1, force_recompute=True
)
nc_acc = nc_res['accuracy_mean']
nc_f1 = nc_res['macro_f1_mean']

# Link Prediction
lp_res = run_link_prediction_experiment(
    'cora', method='logistic', use_features=True, use_embeddings=False,
    dimensions=64, n_runs=1, force_recompute=True
)
lp_auc = lp_res['auc_mean']
lp_ap = lp_res['ap_mean']

# Community Detection
partition = run_louvain(G)
true_labels = {n: ds['labels'].iloc[:,0].loc[n] for n in G.nodes()}
cd_metrics = evaluate_communities(true_labels, partition)

results = {
    'dataset': 'cora',
    'node_classification': {
        'model': 'LogisticRegression',
        'accuracy': round(nc_acc, 4),
        'macro_f1': round(nc_f1, 4)
    },
    'link_prediction': {
        'model': 'LogisticRegression (Hadamard)',
        'auc': round(lp_auc, 4),
        'ap': round(lp_ap, 4)
    },
    'community_detection': {
        'method': 'Louvain',
        'nmi': round(cd_metrics['nmi'], 4),
        'ari': round(cd_metrics['ari'], 4),
        'n_communities': len(set(partition.values()))
    }
}

with open(PROJECT_ROOT / 'results' / 'demo_cora.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print('\nРезультаты сохранены в results/')